# AdaBoost from Scratch — Walkthrough

Build a **decision stump** and an **AdaBoost** ensemble by hand, then apply them to the UCI Adult Census Income task.

This notebook mirrors the original 2020 educational write-up, with bug fixes and a cleaner API that matches the `income_classifier` package.


In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from income_classifier.adaboost import (
    calc_alpha,
    calc_epsilon,
    default_weights,
    ent_from_split,
    find_splits,
    simple_binary_tree_fit,
    simple_binary_tree_predict,
    update_weights,
)
from income_classifier.data import load_adult, split_features_target
from income_classifier.pipeline import make_model_pipeline

np.random.seed(42)


## 1. Candidate splits & entropy

For a numeric column, candidate thresholds are midpoints between sorted unique values. Split quality is measured with weighted binary entropy.


In [ ]:
col = np.array([1.0, 2.0, 2.0, 4.0, 5.0])
print("candidates:", find_splits(col))

labels = np.array([0, 0, 0, 1, 1])
print("entropy @ 3.0:", round(ent_from_split(col, 3.0, labels), 4))
print("entropy @ 1.5:", round(ent_from_split(col, 1.5, labels), 4))


## 2. Fit a single stump

`simple_binary_tree_fit` searches every feature/threshold pair and returns the lowest-entropy cut plus majority-class predictions for each side.


In [ ]:
pts = np.array([
    [.5, 3, 1], [1, 2, 1], [3, .5, 0], [2, 3, 0], [3, 4, 1],
    [3.5, 2.5, 0], [3.6, 4.7, 1], [4, 4.2, 1], [4.5, 2, 0], [4.7, 4.5, 0],
])
X_demo, y_demo = pts[:, :2], pts[:, 2].astype(int)

col_idx, split, left, right = simple_binary_tree_fit(X_demo, y_demo)
preds = simple_binary_tree_predict(X_demo, col_idx, split, left, right)
print({"feature": col_idx, "split": split, "left": left, "right": right})
print("stump accuracy:", (preds == y_demo).mean())


## 3. AdaBoost weight updates

\[
\varepsilon_t=\sum_i w_t(i)\mathbb{1}\{y_i\neq f_t(x_i)\},\quad
\alpha_t=\frac12\ln\frac{1-\varepsilon_t}{\varepsilon_t},\quad
w_{t+1}(i)\propto w_t(i)\,e^{-\alpha_t y_i f_t(x_i)}
\]


In [ ]:
y_true = np.array([1, 0, 1, 0])
y_pred = np.array([1, 0, 0, 0])  # error on index 2
w = default_weights(4)
eps = calc_epsilon(y_true, y_pred, w)
alpha = calc_alpha(eps)
w2 = update_weights(w, alpha, y_true, y_pred)
print({"epsilon": eps, "alpha": round(alpha, 4)})
print("updated weights:", np.round(w2, 4))


## 4. `SimpleAdaBoost` on Adult Census Income

Use the packaged preprocessor + from-scratch booster (sklearn depth-1 stumps under the hood for speed).


In [ ]:
X, y = split_features_target(load_adult())
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

pipe = make_model_pipeline("adaboost_from_scratch", random_state=42)
pipe.fit(X_train, y_train)
pred = pipe.predict(X_test)
print(classification_report(y_test, pred, digits=4))


## 5. Compare with sklearn AdaBoost

The educational implementation should land in the same accuracy band as `AdaBoostClassifier`.


In [ ]:
sk = make_model_pipeline("adaboost_sklearn", random_state=42)
sk.fit(X_train, y_train)
print("from-scratch accuracy:", round(pipe.score(X_test, y_test), 4))
print("sklearn accuracy:     ", round(sk.score(X_test, y_test), 4))
